In [2]:
!pip install pytorch_tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.8 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import torch
from pytorch_tabnet.tab_model import TabNetClassifier

In [5]:
df = pd.read_csv('/kaggle/input/dgfers/cyber_attack.csv')

In [ ]:
def prepare_perfect_data(df):
    df.columns = df.columns.str.lower()
    merge_classes = ['fuzzers', 'dos', 'reconnaissance', 'analysis', 'backdoor', 'shellcode', 'worms']
    df['attack_cat_merged'] = df['attack_cat'].str.lower().replace(merge_classes, 'other')

    target_mapping = {'exploits': 0, 'generic': 1, 'normal': 2, 'other': 3}
    y = df['attack_cat_merged'].map(target_mapping).values
    
    cols_to_drop = ['attack_cat', 'attack_cat_merged', 'label', 'id']
    X_df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    cat_cols = ['proto', 'service', 'state']
    cat_idxs = []
    cat_dims = []
    
    for i, col in enumerate(X_df.columns):
        if col in cat_cols:
            le = LabelEncoder()
            X_df[col] = le.fit_transform(X_df[col].astype(str))
            cat_idxs.append(i)
            cat_dims.append(len(le.classes_))
            
    scaler = StandardScaler()
    num_cols = [c for c in X_df.columns if c not in cat_cols]
    X_df[num_cols] = scaler.fit_transform(X_df[num_cols])
    
    return X_df.values, y, cat_idxs, cat_dims

In [ ]:
X, y, cat_idxs, cat_dims = prepare_perfect_data(df)

In [ ]:
smote = SMOTE(sampling_strategy='not majority', random_state=42)
X_res, y_res = smote.fit_resample(X, y)

In [ ]:
tabnet = TabNetClassifier(
    n_d=128, n_a=128,      # Large memory capacity
    n_steps=10,            # High complexity steps
    gamma=1.3,             # Feature re-usage factor
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=10,
    lambda_sparse=0,       # No penalty for using all features
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=1e-2),
    mask_type='entmax',
    device_name='cuda'
)

In [ ]:
tabnet.fit(
    X_train=X_res, y_train=y_res,
    max_epochs=200,
    batch_size=8192,
    virtual_batch_size=512,
    patience=0,           
    drop_last=False
)

final_preds = tabnet.predict(X)
print(f"FINAL ACCURACY: {accuracy_score(y, final_preds)*100:.2f}%")
print(classification_report(y, final_preds))

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.70586 |  0:00:11s
epoch 1  | loss: 0.05912 |  0:00:22s
epoch 2  | loss: 0.01903 |  0:00:33s
epoch 3  | loss: 0.00636 |  0:00:44s
epoch 4  | loss: 0.00161 |  0:00:55s
epoch 5  | loss: 0.00126 |  0:01:05s
epoch 6  | loss: 0.00116 |  0:01:16s
epoch 7  | loss: 0.00118 |  0:01:27s
epoch 8  | loss: 0.00089 |  0:01:38s
epoch 9  | loss: 0.00163 |  0:01:48s
epoch 10 | loss: 0.00224 |  0:01:59s
epoch 11 | loss: 0.00139 |  0:02:10s
epoch 12 | loss: 0.00192 |  0:02:21s
epoch 13 | loss: 0.00136 |  0:02:32s
epoch 14 | loss: 0.00114 |  0:02:43s
epoch 15 | loss: 0.00129 |  0:02:54s
epoch 16 | loss: 0.00272 |  0:03:05s
epoch 17 | loss: 0.00613 |  0:03:15s
epoch 18 | loss: 0.00195 |  0:03:26s
epoch 19 | loss: 0.00124 |  0:03:37s
epoch 20 | loss: 0.00084 |  0:03:48s
epoch 21 | loss: 0.00096 |  0:03:59s
epoch 22 | loss: 0.00137 |  0:04:10s
epoch 23 | loss: 0.0033  |  0:04:21s
epoch 24 | loss: 0.0015  |  0:04:32s
epoch 25 | loss: 0.00393 |  0:04:43s
epoch 26 | loss: 0.00305 |  0:04:54s
e